In [11]:
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import qualitative

# Color palette (consistent with codebase)
colors = qualitative.G10

# Flow Control Metrics Visualization

Analysis of flow control experiment data from `flow_control_metrics.csv`, extracted from the `simulator-epp-flow-control.py` script run in a custom pod.

## Load Data

In [12]:
# Load the CSV data
df = pd.read_csv('flow_control_metrics.csv')

# Display basic information
print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nUnique tenants: {sorted(df['tenant'].unique())}")
print(f"\nTime range: {df['t'].min():.1f}s to {df['t'].max():.1f}s")

# Dictionary to store figures for export
figures = {}

# Show first few rows
df.head(10)

Data shape: (1508, 11)

Columns: ['t', 'tenant', 'target_qps', 'p90_ttft', 'p90_dur', 'achieved_qps', 's_200', 's_429', 's_503', 's_err', 'active_concurrency']

Unique tenants: ['batch-A', 'premium-A', 'standard-A', 'standard-B']

Time range: 0.0s to 188.0s


,t,tenant,target_qps,p90_ttft,p90_dur,achieved_qps,s_200,s_429,s_503,s_err,active_concurrency
0,0.0,premium-A,1.276596,NaN,NaN,0.0,0,0,0,0,0
1,0.0,standard-A,2.978723,NaN,NaN,0.0,0,0,0,0,0
2,0.0,standard-B,0.000000,NaN,NaN,0.0,0,0,0,0,0
3,0.0,batch-A,0.000000,NaN,NaN,0.0,0,0,0,0,0
4,0.5,premium-A,1.276596,NaN,NaN,0.0,0,0,0,0,0
5,0.5,standard-A,2.978723,NaN,NaN,0.0,0,0,0,0,1
6,0.5,standard-B,0.000000,NaN,NaN,0.0,0,0,0,0,0
7,0.5,batch-A,0.000000,NaN,NaN,0.0,0,0,0,0,0
8,1.0,premium-A,1.276596,NaN,NaN,0.0,0,0,0,0,0
9,1.0,standard-A,2.978723,NaN,NaN,0.0,0,0,0,0,3


## Workload Chart

Stacked area chart showing target QPS over time for each tenant.

In [18]:
# Create stacked area chart for Workload
fig = go.Figure()

# Sort tenants for consistent ordering
tenants = sorted(df['tenant'].unique())

# Add a trace for each tenant
for i, tenant in enumerate(tenants):
    tenant_data = df[df['tenant'] == tenant].sort_values('t')
    
    fig.add_trace(go.Scatter(
        x=tenant_data['t'],
        y=tenant_data['target_qps'],
        name=tenant,
        #stackgroup='one',
        mode='lines',
        line=dict(width=2, color=colors[i % len(colors)]),
        fill='none'
    ))

# Update layout with consistent styling
fig.update_layout(
    title="Workload",
    xaxis_title="Time (s)",
    yaxis_title="Target QPS",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    height=500
)

# Store figure for export
figures['workload'] = fig

fig.show()

## P90 TTFT (Time to First Token)

Line chart showing the 90th percentile time to first token for each tenant.

In [14]:
# Create line chart for P90 TTFT
fig = go.Figure()

# Add a trace for each tenant
for i, tenant in enumerate(tenants):
    tenant_data = df[df['tenant'] == tenant].sort_values('t')
    # Filter out NaN values for p90_ttft
    tenant_data_filtered = tenant_data[tenant_data['p90_ttft'].notna()]
    
    fig.add_trace(go.Scatter(
        x=tenant_data_filtered['t'],
        y=tenant_data_filtered['p90_ttft'],
        name=tenant,
        mode='lines',
        line=dict(width=2, color=colors[i % len(colors)])
    ))

# Update layout
fig.update_layout(
    title="P90 TTFT (Time to First Token)",
    xaxis_title="Time (s)",
    yaxis_title="P90 TTFT (s)",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    height=500
)

# Store figure for export
figures['p90_ttft'] = fig

fig.show()

## P90 Duration

Line chart showing the 90th percentile request duration for each tenant.

In [15]:
# Create line chart for P90 Duration
fig = go.Figure()

# Add a trace for each tenant
for i, tenant in enumerate(tenants):
    tenant_data = df[df['tenant'] == tenant].sort_values('t')
    # Filter out NaN values for p90_dur
    tenant_data_filtered = tenant_data[tenant_data['p90_dur'].notna()]
    
    fig.add_trace(go.Scatter(
        x=tenant_data_filtered['t'],
        y=tenant_data_filtered['p90_dur'],
        name=tenant,
        mode='lines',
        line=dict(width=2, color=colors[i % len(colors)])
    ))

# Update layout
fig.update_layout(
    title="P90 Duration",
    xaxis_title="Time (s)",
    yaxis_title="P90 Duration (s)",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    height=500
)

# Store figure for export
figures['p90_duration'] = fig

fig.show()

## Status Codes (200 vs 503)

Chart showing successful requests (s_200) and service unavailable errors (s_503) for each tenant.

In [16]:
# Create stacked chart for s_200 and s_503
fig = go.Figure()

# Add traces for each tenant - both s_200 and s_503
for i, tenant in enumerate(tenants):
    tenant_data = df[df['tenant'] == tenant].sort_values('t')
    color = colors[i % len(colors)]
    
    # Create marker size array - show marker every 10 data points
    marker_sizes_200 = [8 if idx % 10 == 0 else 0 for idx in range(len(tenant_data))]
    marker_sizes_503 = [8 if idx % 10 == 0 else 0 for idx in range(len(tenant_data))]
    
    # s_200 trace (circle markers) - stacked
    fig.add_trace(go.Scatter(
        x=tenant_data['t'],
        y=tenant_data['s_200'],
        name=f"{tenant} (200)",
        mode='lines+markers',
        line=dict(width=2, color=color),
        marker=dict(
            size=marker_sizes_200,
            symbol='circle',
            color=color
        ),
        stackgroup='status',
        legendgroup=tenant,
    ))
    
    # s_503 trace (cross markers) - stacked
    fig.add_trace(go.Scatter(
        x=tenant_data['t'],
        y=tenant_data['s_503'],
        name=f"{tenant} (503)",
        mode='lines+markers',
        line=dict(width=2, color=color, dash='dash'),
        marker=dict(
            size=marker_sizes_503,
            symbol='x',
            color=color
        ),
        stackgroup='status',
        legendgroup=tenant,
    ))

# Update layout
fig.update_layout(
    title="Status Codes (200 vs 503)",
    xaxis_title="Time (s)",
    yaxis_title="Count",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    height=500
)

# Store figure for export
figures['status_codes'] = fig

fig.show()

## Export Charts to HTML

Save all charts as interactive HTML files.

In [29]:
# Export all figures to a single HTML file with individual legends
import os

output_dir = '_out/flow-control-charts'
os.makedirs(output_dir, exist_ok=True)

# Generate HTML that embeds each chart separately
html_parts = ["""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Flow Control Charts</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 20px;
            background-color: #ffffff;
        }
        .chart-container {
            margin-bottom: 40px;
        }
    </style>
</head>
<body>
    <h1>Flow Control Charts</h1>
"""]

chart_order = ['workload', 'p90_ttft', 'p90_duration', 'status_codes']
for i, chart_name in enumerate(chart_order):
    fig = figures[chart_name]
    # Get the chart as HTML div (include plotlyjs only for first chart)
    include_js = 'cdn' if i == 0 else False
    chart_html = fig.to_html(include_plotlyjs=include_js, full_html=False, div_id=f'chart_{i}')
    html_parts.append(f'<div class="chart-container">{chart_html}</div>')

html_parts.append("""
</body>
</html>
""")

# Write combined HTML
output_file = os.path.join(output_dir, 'flow_control_charts.html')
with open(output_file, 'w') as f:
    f.write('\n'.join(html_parts))

print(f"Exported all charts to: {output_file}")

Exported all charts to: _out/flow-control-charts/flow_control_charts.html


## Additional Charts

_(More charts to be added here)_